# DwD Supervisor Agent — Deploy Notebook

Run cells top-to-bottom from a Databricks notebook attached to **serverless** or a **single-user** cluster. ~5-10 minutes end-to-end (most of that is the serving endpoint becoming READY).

**Before you start** — check the Databricks UI for:
- A foundation-model serving endpoint you have `CAN_USE` on (default uses `databricks-meta-llama-3.1-405b-instruct`)
- A Unity Catalog catalog/schema you have `CAN_MANAGE` on
- That a serving endpoint named `dwd-supervisor-agent` doesn't already exist (this script will fail otherwise — change `ENDPOINT_NAME` below if so)

**Architecture note:** `agent.py` builds the supervisor manually with `langgraph.graph.StateGraph` instead of `langgraph.prebuilt.create_react_agent`. The prebuilt module had a long-running cross-version-skew issue (`tool_node.py` importing symbols missing from `langgraph.runtime`); the manual graph uses only the stable `langgraph.graph` API which has been unchanged across releases.

## 1. Install dependencies (clean install)

First uninstall any pre-baked langgraph/langchain that the Databricks runtime ships with — these can shadow the requirements.txt pins via pip's resolver. Then install fresh from `requirements.txt`.

Restarts the Python interpreter at the end. Re-run subsequent cells after the restart.

In [0]:
%pip uninstall -y -q langgraph langgraph-prebuilt langgraph-checkpoint langgraph-sdk langchain langchain-core langchain-community 2>/dev/null
%pip install -q --no-cache-dir -r requirements.txt
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 2. Configure — EDIT THIS CELL

Defaults below match the bundled DwD demo (`proxy/config.json` Genie spaces). Change any value you need to before running.

For production, replace `os.environ[...] = ...` with `dbutils.secrets.get(scope, key)` and store credentials in a Databricks secret scope.

In [ ]:
import os

# Genie space IDs (find them in the Genie space URL)
os.environ['SALES_SPACE_ID']    = '01f13fe275b515608abb4182d0a37373'
os.environ['CUSTOMER_SPACE_ID'] = '01f13fe2c42f13c4a8f80254aab32c0f'
os.environ['OPS_SPACE_ID']      = '01f13fe2ca67114cbe3d056765823125'
os.environ['HSE_SPACE_ID']      = '01f13d2bcd0a1be2a333d78bca0911b6'

# Foundation model that backs the supervisor's reasoning + tool routing.
# Was llama-3.1-405b but its per-workspace token-per-minute quota is too
# tight for multi-tool composite questions (each agent turn can burn
# 10-50K tokens across the reasoning loop). gpt-oss-120b has much higher
# rate limits and comparable quality for routing + synthesis.
# Available alternatives in this workspace:
#   databricks-meta-llama-3.1-405b-instruct (smartest, tight limits)
#   databricks-gpt-oss-120b                 (recommended for agent loops)
#   databricks-gpt-oss-20b                  (faster, less capable)
#   databricks-qwen3-next-80b-a3b-instruct  (alternative)
#   databricks-gemma-3-12b                  (smallest)
os.environ['SUPERVISOR_LLM_ENDPOINT'] = 'databricks-gpt-oss-120b'

# Where to register the agent in Unity Catalog
os.environ['UC_CATALOG'] = 'workspace'
os.environ['UC_SCHEMA']  = 'databrickspractice'
os.environ['AGENT_NAME'] = 'dwd_supervisor_agent'

# Mosaic AI serving endpoint name (must be unique in this workspace)
os.environ['ENDPOINT_NAME'] = 'dwd-supervisor-agent'

# Explicit Databricks creds for the deployed endpoint to use when calling
# Genie spaces. agent.py reads DATABRICKS_HOST/TOKEN and constructs a
# WorkspaceClient — needed because Mosaic AI's automatic OBO auth doesn't
# reliably propagate CAN_RUN to user-owned Genie spaces.
#
# DO NOT commit a real PAT into this notebook. Two safe ways to provide it:
#
#   (A) Databricks secret scope (production / shared notebooks):
#       os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get('<scope>', '<key>')
#
#   (B) Inline edit only on your local notebook session (demo / dev):
#       Uncomment + paste your PAT below, then NEVER save+commit this cell.
#       Keep it in your local notebook only; the version in git stays empty.
#
os.environ['DATABRICKS_HOST']  = 'https://dbc-f88d29ce-4aa2.cloud.databricks.com'
# os.environ['DATABRICKS_TOKEN'] = 'dapi...'              # ← inline (option B)
# os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get('dwd', 'pat')  # ← scope (option A)

if not os.environ.get('DATABRICKS_TOKEN'):
    raise RuntimeError(
        'DATABRICKS_TOKEN not set. Either uncomment one of the two lines above '
        'with your real PAT (option B) or wire a secret scope (option A) before '
        'continuing. The deployed agent needs this to call your Genie spaces.'
    )

print('Config set.')
print('  Spaces:    ', {k: os.environ[k][:8] + '...' for k in ['SALES_SPACE_ID', 'CUSTOMER_SPACE_ID', 'OPS_SPACE_ID', 'HSE_SPACE_ID']})
print('  LLM:       ', os.environ['SUPERVISOR_LLM_ENDPOINT'])
print('  UC target: ', f"{os.environ['UC_CATALOG']}.{os.environ['UC_SCHEMA']}.{os.environ['AGENT_NAME']}")
print('  Endpoint:  ', os.environ['ENDPOINT_NAME'])
print('  Auth:      ', f"{os.environ['DATABRICKS_HOST']} (token={os.environ['DATABRICKS_TOKEN'][:8]}...)")

## 3a. Sanity-check the install

Quick import check before the slower smoke test below. If this cell fails, the install in cell 1 didn't take and the smoke test will hang for minutes before erroring — better to know immediately.

In [0]:
from importlib.metadata import version, PackageNotFoundError
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from databricks_langchain import ChatDatabricks
try:
    from databricks_langchain import GenieAgent
except ImportError:
    from databricks_langchain.genie import GenieAgent
for pkg in ['mlflow', 'langgraph', 'langchain', 'langchain-core', 'databricks-langchain', 'databricks-agents']:
    try:
        print(f'{pkg:<25} {version(pkg)}')
    except PackageNotFoundError:
        print(f'{pkg:<25} (not installed)')
print('All imports OK.')

mlflow                    3.11.1
langgraph                 1.0.10
langchain                 1.2.10
langchain-core            1.3.2
databricks-langchain      0.19.0
databricks-agents         1.10.0
All imports OK.


## 3b. Smoke-test the agent locally

Instantiate the StateGraph in this notebook and ask it one question. If this fails, the deployed endpoint will fail too — fix the error here first.

Expect ~10-30 seconds (each Genie space the supervisor picks adds latency).

In [0]:
from agent import agent
from mlflow.types.agent import ChatAgentMessage

# ChatAgent exposes .predict(messages, ...) — not .invoke() like a runnable.
response = agent.predict(
    messages=[ChatAgentMessage(role='user', content='Summarise sales and on-time rate for the latest year.')]
)

# ChatAgentResponse.messages[-1] is the assistant's final answer.
print(response.messages[-1].content)

For the latest year, our sales performance was $733,215.26 in total sales, with a profit of $93,439.27 and a profit margin of 12.74%. Our order count was 1,687, with a total quantity sold of 12,476 units.

Our on-time rate for the same period was 79.93%.


[Trace(trace_id=tr-5f4b21fff9fa4222288f6d9020a3d02a), Trace(trace_id=tr-d5f25d2c72e5b4258544ea8aafecb953), Trace(trace_id=tr-a29d84bf86459f5ca94a88a470817da2), Trace(trace_id=tr-03ab58625dc0acc51cc1ec9377afd4ce), Trace(trace_id=tr-b9d4084ea82a0acb861210f384691208), Trace(trace_id=tr-9e66fc01e86e698799559e97e89b0fa8)]

## 4. Log + register + deploy

Runs `log_and_deploy.py`. Creates an MLflow run, registers the model in Unity Catalog, and dispatches a Mosaic AI serving-endpoint deployment.

The endpoint takes ~5-10 minutes to reach READY after this cell returns. Watch the Databricks UI under **Serving** for status.

In [ ]:
import os
import mlflow
# Resources API moved between modules across versions. Try the canonical
# mlflow path first (mlflow >= 2.16), fall back to databricks-agents
# re-export if mlflow is older.
try:
    from mlflow.models.resources import DatabricksGenieSpace, DatabricksServingEndpoint
except ImportError:
    from databricks.agents.resources import DatabricksGenieSpace, DatabricksServingEndpoint
from databricks import agents

# Triggers a load + signature inference of agent.py before logging.
from agent import agent  # noqa: F401

UC_CATALOG = os.environ['UC_CATALOG']
UC_SCHEMA  = os.environ['UC_SCHEMA']
AGENT_NAME = os.environ.get('AGENT_NAME', 'dwd_supervisor_agent')
ENDPOINT_NAME = os.environ.get('ENDPOINT_NAME', 'dwd-supervisor-agent')

UC_MODEL_NAME = f'{UC_CATALOG}.{UC_SCHEMA}.{AGENT_NAME}'
mlflow.set_registry_uri('databricks-uc')

# Resources tell agents.deploy() what to wire OBO auth for. Even when this
# isn't enough on its own (Databricks managed-identity propagation can be
# inconsistent for Genie spaces), declaring them is still good practice.
resources = [
    DatabricksGenieSpace(genie_space_id=os.environ['SALES_SPACE_ID']),
    DatabricksGenieSpace(genie_space_id=os.environ['CUSTOMER_SPACE_ID']),
    DatabricksGenieSpace(genie_space_id=os.environ['OPS_SPACE_ID']),
    DatabricksGenieSpace(genie_space_id=os.environ['HSE_SPACE_ID']),
    DatabricksServingEndpoint(endpoint_name=os.environ['SUPERVISOR_LLM_ENDPOINT']),
]

with mlflow.start_run(run_name='dwd-supervisor-agent'):
    logged_agent_info = mlflow.pyfunc.log_model(
        name='agent',
        python_model='agent.py',
        registered_model_name=UC_MODEL_NAME,
        input_example={
            'messages': [
                {
                    'role': 'user',
                    'content': 'Summarise sales, returns, and on-time rate for the latest year.',
                }
            ]
        },
        pip_requirements=[
            'mlflow>=2.20',
            'databricks-langchain',
            'databricks-agents',
        ],
        resources=resources,
    )

print(f'Logged: {logged_agent_info.model_uri}')
print(f'UC: {UC_MODEL_NAME} v{logged_agent_info.registered_model_version}')

# Endpoint env vars. DATABRICKS_HOST/TOKEN are explicit creds the agent.py
# code uses to construct WorkspaceClient — Option 3 (PAT injection). Use a
# dedicated service-principal token in production; user PAT is fine for
# the demo. Token rotates on the user's Databricks profile every 14 days.
deployment = agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=logged_agent_info.registered_model_version,
    endpoint_name=ENDPOINT_NAME,
    scale_to_zero=True,
    environment_vars={
        'SALES_SPACE_ID':    os.environ['SALES_SPACE_ID'],
        'CUSTOMER_SPACE_ID': os.environ['CUSTOMER_SPACE_ID'],
        'OPS_SPACE_ID':      os.environ['OPS_SPACE_ID'],
        'HSE_SPACE_ID':      os.environ['HSE_SPACE_ID'],
        'SUPERVISOR_LLM_ENDPOINT': os.environ.get(
            'SUPERVISOR_LLM_ENDPOINT', 'databricks-meta-llama-3.1-405b-instruct'
        ),
        # Explicit auth — agent.py uses these to build a WorkspaceClient
        # that has CAN_RUN on the Genie spaces (because they're the user's
        # own creds). Pulled from os.environ so the deploying notebook
        # decides what creds to bake in.
        'DATABRICKS_HOST':  os.environ.get('DATABRICKS_HOST', 'https://dbc-f88d29ce-4aa2.cloud.databricks.com'),
        'DATABRICKS_TOKEN': os.environ.get('DATABRICKS_TOKEN', os.environ.get('DBR_PAT', '')),
    },
)

print()
print(f'Deployment dispatched. Endpoint will become READY in ~5-10 minutes.')
print(f'Endpoint URL: {deployment.endpoint_url}')

## 5. Wait until READY, then test the endpoint

Polls the endpoint state until it's READY (or fails). Then sends a test invocation directly to verify the deployed agent works end-to-end.

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole, EndpointStateReady
from databricks.sdk.errors import ResourceDoesNotExist

w = WorkspaceClient()
endpoint_name = os.environ['ENDPOINT_NAME']

# Endpoint may take a few seconds to appear in the API after deploy()
# returns. Retry a handful of times before giving up.
for attempt in range(10):
    try:
        ep = w.serving_endpoints.get(endpoint_name)
        break
    except ResourceDoesNotExist:
        if attempt < 9:
            print(f'[{attempt*5:>4}s] Endpoint not yet visible, retrying...')
            time.sleep(5)
        else:
            raise

# Poll state until READY (or fail). Cold-start typically 5-10 minutes.
for i in range(60):
    ep = w.serving_endpoints.get(endpoint_name)
    state = ep.state.ready if ep.state else None
    print(f'[{i*30:>4}s] state={state}')
    if state == EndpointStateReady.READY:
        response = w.serving_endpoints.query(
            name=endpoint_name,
            messages=[
                ChatMessage(
                    role=ChatMessageRole.USER,
                    content='What were total sales and profit margin for the latest year?',
                )
            ],
        )
        print('\n--- Endpoint response ---')
        print(response.choices[0].message.content if response.choices else response)
        break
    if state and 'FAILED' in str(state):
        raise RuntimeError(f'Endpoint deployment failed: {ep.state}')
    time.sleep(30)

[   0s] state=EndpointStateReady.NOT_READY
[  30s] state=EndpointStateReady.NOT_READY
[  60s] state=EndpointStateReady.NOT_READY
[  90s] state=EndpointStateReady.NOT_READY
[ 120s] state=EndpointStateReady.NOT_READY
[ 150s] state=EndpointStateReady.NOT_READY
[ 180s] state=EndpointStateReady.NOT_READY
[ 210s] state=EndpointStateReady.NOT_READY
[ 240s] state=EndpointStateReady.NOT_READY
[ 270s] state=EndpointStateReady.NOT_READY
[ 300s] state=EndpointStateReady.NOT_READY
[ 330s] state=EndpointStateReady.NOT_READY
[ 360s] state=EndpointStateReady.NOT_READY
[ 390s] state=EndpointStateReady.NOT_READY
[ 420s] state=EndpointStateReady.NOT_READY
[ 450s] state=EndpointStateReady.NOT_READY
[ 480s] state=EndpointStateReady.NOT_READY
[ 510s] state=EndpointStateReady.NOT_READY
[ 540s] state=EndpointStateReady.NOT_READY
[ 570s] state=EndpointStateReady.NOT_READY
[ 600s] state=EndpointStateReady.NOT_READY
[ 630s] state=EndpointStateReady.NOT_READY
[ 660s] state=EndpointStateReady.NOT_READY
[ 690s] sta

## 6. Wire the proxy to the new endpoint

Copy the JSON snippet below into `proxy/config.json`, replacing the existing `supervisor` profile. Then restart the proxy:

```powershell
cd proxy ; node server.js
```

In Power BI, switch the visual's connection mode to **Supervisor** and ask a cross-domain question. Watch the MLflow trace UI in Databricks to see which Genie spaces the agent chose to call.

In [0]:
workspace_host = w.config.host.rstrip('/')
snippet = f'''  "supervisor": {{
    "type": "supervisor",
    "host": "{workspace_host}",
    "endpoint": "/serving-endpoints/{endpoint_name}/invocations",
    "agentName": "DwD Supervisor Agent",
    "token": "<PAT or service-principal token with CAN_USE on the endpoint>",
    "displayName": "DwD Supervisor Agent",
    "dataDomain": "all helper data"
  }}'''
print(snippet)

  "supervisor": {
    "type": "supervisor",
    "host": "https://dbc-f88d29ce-4aa2.cloud.databricks.com",
    "endpoint": "/serving-endpoints/dwd-supervisor-agent/invocations",
    "agentName": "DwD Supervisor Agent",
    "token": "<PAT or service-principal token with CAN_USE on the endpoint>",
    "displayName": "DwD Supervisor Agent",
    "dataDomain": "all helper data"
  }
